# Multi-Asset Goal-Reaching HJB with Min Dai Viscosity Correction

This notebook implements a **wealth-only finite-difference HJB solver** for a multi-asset goal-reaching problem.

We solve
$$
V(t,w)=\sup_{\pi\in\mathcal A}\mathbb P(W_T\ge 1\mid W_t=w),
$$
with wealth dynamics
$$
dW_t = W_t\,\pi_t^\top(\mu-r\mathbf 1)dt + W_t\,\pi_t^\top \Sigma^{1/2} dB_t,
$$
and terminal utility
$$
U(W_T)=\mathbf 1_{\{W_T\ge 1\}}.
$$

The HJB remains **1D in wealth**:
$$
V_t + \sup_{\pi\in\mathcal A}\left[w(\pi^\top(\mu-r\mathbf 1))V_w + \tfrac12 w^2(\pi^\top\Sigma\pi)V_{ww}\right]=0,
$$
but the control is an $n$-vector of portfolio weights.

Near terminal time we use the **Min Dai viscosity / asymptotic terminal adjustment** (Eq. 18 idea):
$$
V(t,w) \approx 2\Phi\left(\frac{\min\{0,\log(w/1)\}}{\sigma_*\sqrt{\tau}}\right),\qquad \tau=T-t,
$$
where
$$
\sigma_* = \sup_{\pi\in\mathcal A}\sqrt{\pi^\top\Sigma\pi}.
$$
This is the multi-asset analogue of the 1D terminal-time correction.

In [17]:
import json
import math
import time
from pathlib import Path

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np

np.set_printoptions(precision=4, suppress=True)
plt.rcParams.update({'figure.dpi': 140, 'font.size': 10})

In [18]:
def _normcdf_local(z):
    z = np.asarray(z, dtype=float)
    return 0.5 * (1.0 + np.vectorize(math.erf)(z / math.sqrt(2.0)))


def terminal_goal_utility(w_arr, goal=1.0):
    return (np.asarray(w_arr, dtype=float) >= goal).astype(float)


def project_weights_local(weights, d, u, leverage_limit=1.0, long_only=True, allow_cash=True):
    projected = np.clip(np.asarray(weights, dtype=float), d, u)
    gross = float(np.sum(np.abs(projected)))
    if gross > leverage_limit and gross > 0.0:
        projected = projected / gross * leverage_limit
    total = float(projected.sum())
    if long_only:
        total = max(total, 1e-12)
        if total > 1.0 or not allow_cash:
            projected = projected / total
    return projected


def sample_admissible_controls(n_assets, d, u, n_samples=256, leverage_limit=1.0,
                              long_only=True, allow_cash=True, seed=0):
    rng = np.random.default_rng(seed)
    base = rng.dirichlet(np.ones(n_assets), size=n_samples)
    if not long_only:
        base = base * rng.choice([-1.0, 1.0], size=base.shape)
    anchors = [np.zeros(n_assets)]
    anchors.extend(np.eye(n_assets))
    anchors.append(np.full(n_assets, 1.0 / n_assets))
    controls = np.vstack([np.asarray(anchors), base])
    projected = [
        project_weights_local(ctrl, d, u, leverage_limit=leverage_limit,
                              long_only=long_only, allow_cash=allow_cash)
        for ctrl in controls
    ]
    return np.unique(np.round(np.vstack(projected), decimals=12), axis=0)


def sigma_star_from_controls(covariance, controls):
    sig2 = np.einsum('ij,jk,ik->i', controls, covariance, controls)
    return float(np.sqrt(np.max(np.maximum(sig2, 1e-16))))


def asymptotic_V_goalreach_multiasset(w_arr, tau, covariance, controls, goal=1.0):
    tau = max(float(tau), 1e-12)
    sigma_star = max(sigma_star_from_controls(covariance, controls), 1e-12)
    z = np.minimum(0.0, np.log(np.maximum(w_arr, 1e-12) / goal)) / (sigma_star * np.sqrt(tau))
    return 2.0 * _normcdf_local(z)


def generate_market_params(n_assets, seed=42):
    rng = np.random.default_rng(seed)
    r = 0.03
    market_loading = rng.uniform(0.4, 1.0, size=n_assets)
    market_vol = 0.18
    idio_vol = rng.uniform(0.10, 0.22, size=n_assets)
    covariance = market_vol**2 * np.outer(market_loading, market_loading) + np.diag(idio_vol**2)
    excess_mu = rng.uniform(0.05, 0.12, size=n_assets)
    return excess_mu + r, r, covariance

In [19]:
def _thomas_solve(a, b, c, rhs):
    n = len(b)
    c2 = np.zeros(n)
    d2 = np.zeros(n)
    x = np.zeros(n)
    c2[0] = c[0] / b[0]
    d2[0] = rhs[0] / b[0]
    for k in range(1, n):
        den = b[k] - a[k] * c2[k - 1]
        c2[k] = c[k] / den if k < n - 1 else 0.0
        d2[k] = (rhs[k] - a[k] * d2[k - 1]) / den
    x[-1] = d2[-1]
    for k in range(n - 2, -1, -1):
        x[k] = d2[k] - c2[k] * x[k + 1]
    return x


def fd_solve_viscosity_goal_multiasset(mu, r, covariance, T, A, Nw, Nt, d, u,
                                       goal=1.0, UB=0.0, UA=1.0,
                                       tau_asymp=None, record_taus=None,
                                       n_control_samples=256,
                                       leverage_limit=1.0,
                                       long_only=True,
                                       allow_cash=True,
                                       seed=0):
    mu = np.asarray(mu, dtype=float)
    covariance = np.asarray(covariance, dtype=float)
    n_assets = len(mu)
    controls = sample_admissible_controls(
        n_assets=n_assets,
        d=d,
        u=u,
        n_samples=n_control_samples,
        leverage_limit=leverage_limit,
        long_only=long_only,
        allow_cash=allow_cash,
        seed=seed,
    )
    excess_mu = mu - r
    w = np.linspace(0.0, A, Nw + 1)
    dw = A / Nw
    dt = T / Nt
    wi = w[1:Nw]
    if tau_asymp is None:
        tau_asymp = 0.08 * T
    V = terminal_goal_utility(w, goal=goal).astype(float)
    mu_eff = controls @ excess_mu
    sigma_eff = np.sqrt(np.maximum(np.einsum('ij,jk,ik->i', controls, covariance, controls), 1e-16))
    sigma_star = sigma_star_from_controls(covariance, controls)
    snapshots = {}
    policy_snaps = {}
    chosen_controls = np.zeros((Nw + 1, n_assets))

    def browne_V_local(w_arr, tau_val):
        tau_val = max(float(tau_val), 1e-10)
        safe_w = np.maximum(w_arr, 1e-12)
        safe_sig = np.maximum(sigma_eff, 1e-12)
        z = (
            np.log(safe_w[None, :] / goal)
            + (mu_eff[:, None] - 0.5 * safe_sig[:, None] ** 2) * tau_val
        ) / (safe_sig[:, None] * math.sqrt(tau_val))
        return _normcdf_local(z)

    def asymptotic_fn(w_arr, tau_val):
        return asymptotic_V_goalreach_multiasset(w_arr, tau_val, covariance, controls, goal=goal)

    for step in range(Nt):
        tau = (Nt - step) * dt
        V_old = V.copy()
        alpha = float(np.exp(-tau / tau_asymp))
        V_b = browne_V_local(w, tau)
        V_a = np.repeat(asymptotic_fn(w, tau)[None, :], len(controls), axis=0)
        V_ws = alpha * V_a + (1.0 - alpha) * V_b
        best_idx = np.argmax(V_ws, axis=0)[1:Nw]
        chosen_controls[1:Nw] = controls[best_idx]

        for _it in range(60):
            prev_idx = best_idx.copy()
            mu_now = mu_eff[best_idx]
            sig_now = sigma_eff[best_idx]
            a2 = 0.5 * sig_now**2 * wi**2
            adv = mu_now * wi
            ap = np.maximum(adv, 0.0) / dw
            am = np.minimum(adv, 0.0) / dw
            a_s = -dt * (a2 / dw**2 - am)
            b_m = 1.0 + dt * (2.0 * a2 / dw**2 + ap - am)
            c_s = -dt * (a2 / dw**2 + ap)
            rhs = V_old[1:Nw].copy()
            rhs[0] -= a_s[0] * UB
            rhs[-1] -= c_s[-1] * UA
            a_s[0] = 0.0
            c_s[-1] = 0.0
            V_int = _thomas_solve(a_s, b_m, c_s, rhs)
            V_new = np.empty(Nw + 1)
            V_new[0] = UB
            V_new[-1] = UA
            V_new[1:Nw] = V_int
            Vw = (V_new[2:] - V_new[:-2]) / (2.0 * dw)
            Vww = (V_new[2:] - 2.0 * V_new[1:-1] + V_new[:-2]) / dw**2
            H = (
                0.5 * sigma_eff[:, None] ** 2 * wi[None, :] ** 2 * Vww[None, :]
                + mu_eff[:, None] * wi[None, :] * Vw[None, :]
            )
            best_idx = np.argmax(H, axis=0)
            chosen_controls[1:Nw] = controls[best_idx]
            V = V_new
            if np.array_equal(best_idx, prev_idx):
                break

        chosen_controls[0] = controls[0]
        chosen_controls[-1] = controls[best_idx[-1]] if len(best_idx) else controls[0]

        if record_taus:
            tau_prev = (Nt - step + 1) * dt
            for tr in record_taus:
                if tau_prev >= tr >= tau and tr not in snapshots:
                    snapshots[tr] = V.copy()
                    policy_snaps[tr] = chosen_controls.copy()

    if record_taus:
        snapshots[0.0] = V.copy()
        policy_snaps[0.0] = chosen_controls.copy()

    return {
        'wealth_grid': w,
        'value': V,
        'policy': chosen_controls,
        'controls': controls,
        'snapshots': snapshots,
        'policy_snaps': policy_snaps,
        'sigma_star': sigma_star,
    }

In [20]:
TAU_RECORD = [0.0, 0.005, 0.01, 0.02, 0.05, 0.1]
N_VALUES = [1, 5, 10, 20]
NOTEBOOK_RESULTS = {}

for n in N_VALUES:
    mu, r, covariance = generate_market_params(n, seed=100 + n)
    t0 = time.time()
    result = fd_solve_viscosity_goal_multiasset(
        mu=mu,
        r=r,
        covariance=covariance,
        T=1.0,
        A=1.6,
        Nw=100,
        Nt=200,
        d=0.0,
        u=1.0,
        goal=1.0,
        UB=0.0,
        UA=1.0,
        tau_asymp=0.08,
        record_taus=TAU_RECORD,
        n_control_samples=96,
        leverage_limit=1.0,
        long_only=True,
        allow_cash=True,
        seed=11 + n,
    )
    elapsed = time.time() - t0
    w = result['wealth_grid']
    V = result['value']
    Pi = result['policy']
    NOTEBOOK_RESULTS[n] = {
        'elapsed_sec': elapsed,
        'V_0p80': float(np.interp(0.8, w, V)),
        'V_0p90': float(np.interp(0.9, w, V)),
        'V_1p00': float(np.interp(1.0, w, V)),
        'first_policy_above_0p80': Pi[int(np.searchsorted(w, 0.8))].copy(),
        'sigma_star': float(result['sigma_star']),
        'result': result,
    }
    print(
        f"n={n:2d} | time={elapsed:5.2f}s | "
        f"V(0.8)={NOTEBOOK_RESULTS[n]['V_0p80']:.4f} | "
        f"V(0.9)={NOTEBOOK_RESULTS[n]['V_0p90']:.4f} | "
        f"sigma_star={NOTEBOOK_RESULTS[n]['sigma_star']:.4f}"
    )

NOTEBOOK_SMOKE_SUMMARY = {
    n: {
        'V_0p80': NOTEBOOK_RESULTS[n]['V_0p80'],
        'V_0p90': NOTEBOOK_RESULTS[n]['V_0p90'],
        'V_1p00': NOTEBOOK_RESULTS[n]['V_1p00'],
        'policy_sum_0p80': float(NOTEBOOK_RESULTS[n]['first_policy_above_0p80'].sum()),
    }
    for n in N_VALUES
}
NOTEBOOK_SMOKE_SUMMARY

n= 1 | time= 1.14s | V(0.8)=0.4291 | V(0.9)=0.7217 | sigma_star=0.2252
n= 5 | time= 1.88s | V(0.8)=0.4856 | V(0.9)=0.7522 | sigma_star=0.2730
n=10 | time= 1.82s | V(0.8)=0.4621 | V(0.9)=0.7574 | sigma_star=0.2430
n=20 | time= 1.85s | V(0.8)=0.5047 | V(0.9)=0.7792 | sigma_star=0.2624


{1: {'V_0p80': 0.42908371368385295,
  'V_0p90': 0.7216835692190777,
  'V_1p00': 0.9820656338740561,
  'policy_sum_0p80': 1.0},
 5: {'V_0p80': 0.4855533737350459,
  'V_0p90': 0.7522007303334524,
  'V_1p00': 0.9851942714430886,
  'policy_sum_0p80': 1.0},
 10: {'V_0p80': 0.46207444139335485,
  'V_0p90': 0.7574029858983076,
  'V_1p00': 0.9873076764446852,
  'policy_sum_0p80': 1.0},
 20: {'V_0p80': 0.5046996567944423,
  'V_0p90': 0.7792010018770607,
  'V_1p00': 0.9887237999885339,
  'policy_sum_0p80': 1.0}}

In [21]:
n_check = 5
check = NOTEBOOK_RESULTS[n_check]['result']
w = check['wealth_grid']
sigma_star = check['sigma_star']
asymp_errors = {}

for tau in sorted(k for k in check['snapshots'] if k > 0.0):
    fd_val = check['snapshots'][tau]
    asymp = 2.0 * _normcdf_local(
        np.minimum(0.0, np.log(np.maximum(w, 1e-12) / 1.0)) / (sigma_star * np.sqrt(tau))
    )
    asymp_errors[tau] = float(np.max(np.abs(fd_val - asymp)))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for tau in [0.1, 0.02, 0.01, 0.005]:
    if tau not in check['snapshots']:
        continue
    axes[0].plot(w, check['snapshots'][tau], label=f'FD tau={tau}')
    axes[0].plot(
        w,
        2.0 * _normcdf_local(np.minimum(0.0, np.log(np.maximum(w, 1e-12))) / (sigma_star * np.sqrt(tau))),
        '--',
        linewidth=1.2,
    )
axes[0].axvline(1.0, color='gray', linestyle=':', linewidth=0.8)
axes[0].set_title('FD vs Min Dai asymptotic near T (n=5)')
axes[0].set_xlabel('wealth')
axes[0].set_ylabel('value')
axes[0].legend(fontsize=7)

taus = np.array(sorted(asymp_errors))
errs = np.array([asymp_errors[t] for t in taus])
axes[1].loglog(taus, errs, 'o-')
axes[1].set_title('max error to asymptotic')
axes[1].set_xlabel('tau')
axes[1].set_ylabel('max |V - V_asymp|')

out_dir = Path('results/notebook_checks')
out_dir.mkdir(parents=True, exist_ok=True)
fig.tight_layout()
fig.savefig(out_dir / 'multi_asset_viscosity_goalreach_check.png', bbox_inches='tight')
plt.close(fig)

ASYMPTOTIC_ERROR_TABLE = asymp_errors
ASYMPTOTIC_ERROR_TABLE

{0.005: 0.8679143183789095,
 0.01: 0.829310353114757,
 0.02: 0.7699802149074941,
 0.05: 0.6612243370135017,
 0.1: 0.5429651551992427}

In [22]:
fig, axes = plt.subplots(2, 2, figsize=(13, 8), sharex=True, sharey=True)
axes = axes.ravel()

for ax, n in zip(axes, N_VALUES):
    result = NOTEBOOK_RESULTS[n]['result']
    w = result['wealth_grid']
    Pi = result['policy']
    colors = plt.cm.tab20(np.linspace(0, 1, max(n, 2)))[:n]
    ax.stackplot(w, Pi.T, colors=colors, alpha=0.9)
    ax.axvline(1.0, color='black', linestyle=':', linewidth=1.0)
    ax.set_title(f'{n} assets')
    ax.set_xlim(w[0], w[-1])
    ax.set_ylim(0.0, 1.0)
    ax.set_xlabel('wealth')
    ax.set_ylabel('portfolio weight')

handles = [plt.Line2D([0], [0], color=plt.cm.tab20(i / max(19, 1)), lw=6) for i in range(min(10, max(N_VALUES)))]
labels = [f'asset {i + 1}' for i in range(min(10, max(N_VALUES)))]
fig.legend(handles, labels, loc='lower center', ncol=5, frameon=False)
fig.suptitle('Optimal portfolio weights across wealth', fontsize=14)
fig.tight_layout(rect=[0, 0.08, 1, 0.96])
fig.savefig(out_dir / 'multi_asset_policy_weights_across_wealth.png', bbox_inches='tight')
plt.close(fig)
out_dir / 'multi_asset_policy_weights_across_wealth.png'

PosixPath('results/notebook_checks/multi_asset_policy_weights_across_wealth.png')

In [23]:
result = NOTEBOOK_RESULTS[5]['result']
w = result['wealth_grid']
Pi = result['policy']

idx = np.searchsorted(w, 0.8)
print("wealth =", w[idx])
print("optimal weights =", Pi[idx])


wealth = 0.8
optimal weights = [0. 1. 0. 0. 0.]


## Notes

- The FD grid is still **1D in wealth**.
- The multi-asset part appears in the inner HJB optimisation over sampled admissible portfolios.
- The Min Dai terminal-time adjustment enters through the asymptotic warmstart near $T$.
- If this formulation looks right, the next step is to decide whether to fold it into the package solver and retire the older notebook version.